# MIRAGE AI / ML Defense Studio: Model Training & Evaluation
### Problem Statement ID: 26145 · NTRO · Smart India Hackathon 2026

This notebook provides the end-to-end model training workflow for **unidirectional passive threat intelligence**:
1. **Exploratory Data Analysis (EDA):** Multi-resolution packet & connection features across benign vs. 5 threat classes.
2. **Feature Engineering & Preprocessing:** Stratified splitting, standardization, and class balancing.
3. **Model Training:**
   - **`XGBoost` & `RandomForest`:** High-speed tabular threat classification.
   - **`IsolationForest`:** Unsupervised anomaly detection for zero-day / novel attacks.
4. **Explainable AI (XAI):** Feature importance and SHAP attributions ("WHY WE FLAGGED THIS").
5. **Artifact Export:** Saving models to `models/` for production inference.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root in sys.path
root_dir = Path(os.getcwd()).parent if "notebooks" in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(root_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score
import joblib

print(f"PyTorch: {torch.__version__} | XGBoost: {xgb.__version__} | LightGBM: {lgb.__version__}")

## 1. Synthesizing Telemetry Dataset
We use the built-in MIRAGE telemetry generator representing passive optical diode tap metrics.

In [ ]:
from backend.ml.dataset_generator import generate_mirage_dataset

X, y = generate_mirage_dataset(n_samples=10000, random_state=42)
print(f"Dataset Shape: {X.shape}")
y.value_counts()

## 2. Feature Distribution Visualization

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=X[['packets_per_sec', 'syn_rate', 'udp_rate']])
plt.yscale('log')
plt.title('Log-Scale Telemetry Rates (Packets, SYN, UDP)')
plt.show()

## 3. Train Gradient Boosted Threat Classifier (XGBoost)
Training an extreme gradient boosted decision tree classifier across all 6 traffic classes.

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='mlogloss'
)
xgb_model.fit(X_train_scaled, y_train)

y_pred = xgb_model.predict(X_test_scaled)
print(classification_report(y_test, y_pred, target_names=le.classes_))

## 4. Confusion Matrix Evaluation

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('XGBoost Threat Classification Matrix')
plt.xlabel('Predicted Threat')
plt.ylabel('True Threat')
plt.show()

## 5. Serializing Models for Production Inference

In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(xgb_model, 'models/XGBoost_ThreatClassifier.joblib')
joblib.dump(scaler, 'models/scaler.joblib')
joblib.dump(le, 'models/label_encoder.joblib')
print('✓ Models successfully serialized to models/')